# Flute Music → Indian Classical Notes: Strategy Comparison

This notebook tests multiple strategies for converting a piece of flute music into Indian classical swaras (Sa, Re, Ga, Ma, Pa, Dha, Ni and their variants).

## Indian Classical Note System (Swaras)
| Swara | Abbrev | Western (in C) | Freq Ratio | Variants |
|-------|--------|----------------|------------|----------|
| Shadja | Sa | C | 1.000 | Only pure |
| Rishabh | Re | D | 9/8 | Komal Re (C#), Shuddha Re (D) |
| Gandhar | Ga | E | 5/4 | Komal Ga (D#), Shuddha Ga (E) |
| Madhyam | Ma | F | 4/3 | Shuddha Ma (F), Tivra Ma (F#) |
| Pancham | Pa | G | 3/2 | Only pure |
| Dhaivat | Dha | A | 5/3 | Komal Dha (G#), Shuddha Dha (A) |
| Nishad | Ni | B | 15/8 | Komal Ni (A#), Shuddha Ni (B) |

## Strategies Covered
1. **PYIN** — Probabilistic YIN pitch tracker (librosa)
2. **CREPE** — Deep learning pitch estimator (neural network)
3. **Spectral Peak + Harmonic Product Spectrum (HPS)** — Classic DSP approach
4. **Basic-Pitch** — Spotify's ML model (MIDI-level note detection)
5. **Comparison + Voting Ensemble**

## 0. Install Dependencies

In [ ]:
# Run once to install all required packages
!pip install librosa soundfile numpy matplotlib crepe basic-pitch tensorflow scipy --quiet

## 1. Setup & Utilities

In [ ]:
import numpy as np
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import find_peaks
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['font.size'] = 11

In [ ]:
# ── Indian Classical Note Definitions ──────────────────────────────────────────
# All frequency ratios are relative to Sa (tonic).
# We use 12-TET approximations so we can map onto MIDI cents.

SWARAS_12TET = {
    # semitone offset from Sa : (short name, full name, variant)
    0:  ('Sa',    'Shadja',    'shuddha'),
    1:  ('Re♭',   'Rishabh',   'komal'),
    2:  ('Re',    'Rishabh',   'shuddha'),
    3:  ('Ga♭',   'Gandhar',   'komal'),
    4:  ('Ga',    'Gandhar',   'shuddha'),
    5:  ('Ma',    'Madhyam',   'shuddha'),
    6:  ('Ma#',   'Madhyam',   'tivra'),
    7:  ('Pa',    'Pancham',   'shuddha'),
    8:  ('Dha♭',  'Dhaivat',   'komal'),
    9:  ('Dha',   'Dhaivat',   'shuddha'),
    10: ('Ni♭',   'Nishad',    'komal'),
    11: ('Ni',    'Nishad',    'shuddha'),
}

# Colors for each swara (for visualization)
SWARA_COLORS = {
    0: '#FF4444',  1: '#FF8C44',  2: '#FFC044',  3: '#F0E040',
    4: '#88CC44',  5: '#44CC88',  6: '#44CCCC',  7: '#4488FF',
    8: '#8844FF',  9: '#CC44FF', 10: '#FF44CC', 11: '#FF4488',
}

def hz_to_swara(freq_hz, sa_hz=261.63, octave_relative=True):
    """
    Convert a frequency (Hz) to the nearest Indian classical swara.

    Parameters
    ----------
    freq_hz      : float  — input frequency
    sa_hz        : float  — tonic (Sa) frequency in Hz (default C4 = 261.63 Hz)
    octave_relative: bool — if True, collapse all octaves to one saptak

    Returns
    -------
    dict with keys: semitone, short_name, full_name, variant, octave, cents_deviation
    """
    if freq_hz <= 0:
        return None

    # Distance in semitones from Sa
    semitones_from_sa = 12 * np.log2(freq_hz / sa_hz)

    # Octave and position within octave
    octave = int(np.floor(semitones_from_sa / 12))
    semitone_in_octave = semitones_from_sa - (octave * 12)

    # Round to nearest semitone
    nearest = int(round(semitone_in_octave)) % 12
    cents_deviation = (semitone_in_octave - round(semitone_in_octave)) * 100

    short_name, full_name, variant = SWARAS_12TET[nearest]

    octave_label = ['mandra', 'mandra', 'madhya', 'taar'][max(0, min(3, octave + 1))]

    return {
        'semitone':        nearest,
        'short_name':      short_name,
        'full_name':       full_name,
        'variant':         variant,
        'octave':          octave,
        'octave_label':    octave_label,
        'cents_deviation': round(cents_deviation, 1),
        'freq_hz':         round(freq_hz, 2),
    }


def freqs_to_swara_sequence(freqs, times, sa_hz=261.63, min_confidence=0.0):
    """Convert arrays of (times, freqs) to a timestamped swara sequence."""
    result = []
    for t, f in zip(times, freqs):
        if f is None or (hasattr(f, '__float__') and np.isnan(float(f))) or f <= 0:
            result.append({'time': round(float(t), 3), 'swara': None})
        else:
            info = hz_to_swara(float(f), sa_hz=sa_hz)
            if info:
                info['time'] = round(float(t), 3)
            result.append(info)
    return result


def collapse_to_swara_run(swara_sequence, gap_tolerance_s=0.05):
    """
    Collapse a frame-level swara sequence into runs (note events).
    Tiny gaps (< gap_tolerance_s) within the same swara are bridged.
    """
    runs = []
    current = None
    start_t = None

    for item in swara_sequence:
        label = item['short_name'] if item and item.get('short_name') else None
        t = item['time'] if item else None

        if label != current:
            if current is not None:
                runs.append({'swara': current, 'start': start_t, 'end': t})
            current = label
            start_t = t

    if current is not None and start_t is not None:
        runs.append({'swara': current, 'start': start_t, 'end': swara_sequence[-1]['time']})

    return runs


print('Utilities loaded.')
print('Sample mapping — 440 Hz with Sa=261.63 Hz (C4):')
print(hz_to_swara(440.0))

## 2. Load Audio

Replace `AUDIO_FILE` with the path to your flute recording. Any common format works (wav, mp3, flac, ogg).

Set `SA_HZ` to the tonic frequency of the piece — this is the most important parameter for accurate swara mapping.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
AUDIO_FILE = 'your_flute_piece.wav'   # <-- change this path
SA_HZ      = 261.63                   # C4 — change to match actual tonic
SR         = 22050                    # target sample rate

# ── Load ──────────────────────────────────────────────────────────────────────
y, sr = librosa.load(AUDIO_FILE, sr=SR, mono=True)
duration = librosa.get_duration(y=y, sr=sr)

print(f'Loaded: {AUDIO_FILE}')
print(f'Duration : {duration:.2f}s  |  Sample rate: {sr} Hz  |  Samples: {len(y)}')
print(f'Tonic Sa : {SA_HZ} Hz')

# Waveform preview
plt.figure(figsize=(14, 2.5))
librosa.display.waveshow(y, sr=sr, alpha=0.7, color='steelblue')
plt.title('Waveform')
plt.tight_layout()
plt.show()

In [ ]:
# Optional: generate a short synthetic flute-like tone for quick testing
# (comment this block out once you have a real audio file)

def synth_swara_sequence(sa_hz=261.63, sr=22050, note_dur=0.5):
    """Generate a simple ascending Sa-Re-Ga-Ma-Pa-Dha-Ni-Sa sequence."""
    semitones = [0, 2, 4, 5, 7, 9, 11, 12]  # shuddha saptak
    audio = []
    for st in semitones:
        freq = sa_hz * (2 ** (st / 12))
        t = np.linspace(0, note_dur, int(sr * note_dur), endpoint=False)
        # flute-like: fundamental + softened harmonics
        wave  = 0.6 * np.sin(2 * np.pi * freq * t)
        wave += 0.2 * np.sin(2 * np.pi * 2 * freq * t)
        wave += 0.1 * np.sin(2 * np.pi * 3 * freq * t)
        # soft envelope
        env = np.ones_like(t)
        env[:int(0.05*sr)] = np.linspace(0, 1, int(0.05*sr))
        env[-int(0.1*sr):] = np.linspace(1, 0, int(0.1*sr))
        audio.append(wave * env)
    return np.concatenate(audio)

y_synth = synth_swara_sequence(sa_hz=SA_HZ)
sf.write('synth_flute_test.wav', y_synth, SR)

# Uncomment the next two lines to test with the synthetic file
# y, sr = y_synth, SR
# AUDIO_FILE = 'synth_flute_test.wav'

print('Synthetic test file written → synth_flute_test.wav')
print('Uncomment the two lines above to use it.')

## 3. Strategy 1 — PYIN (Probabilistic YIN)

PYIN is a robust time-domain pitch estimator. It gives a voiced/unvoiced decision alongside each frequency estimate, which is very useful for flute (lots of silence between phrases).

In [ ]:
# PYIN pitch tracking
f0_pyin, voiced_flag_pyin, voiced_prob_pyin = librosa.pyin(
    y,
    fmin=librosa.note_to_hz('C2'),   # ~65 Hz
    fmax=librosa.note_to_hz('C7'),   # ~2093 Hz
    sr=sr,
    frame_length=2048,
    hop_length=512,
)

times_pyin = librosa.times_like(f0_pyin, sr=sr, hop_length=512)

# Zero out unvoiced frames
f0_pyin_voiced = np.where(voiced_flag_pyin, f0_pyin, np.nan)

# Convert to swaras
seq_pyin = freqs_to_swara_sequence(
    f0_pyin_voiced, times_pyin, sa_hz=SA_HZ
)
runs_pyin = collapse_to_swara_run(seq_pyin)

print(f'PYIN: {np.sum(voiced_flag_pyin)} voiced frames / {len(voiced_flag_pyin)} total')
print('\nFirst 15 note events:')
for r in runs_pyin[:15]:
    dur = round(r['end'] - r['start'], 3) if r['end'] and r['start'] else '?'
    print(f"  {r['start']:.3f}s – {r['end']:.3f}s  [{dur}s]  → {r['swara']}")

In [ ]:
# Visualize PYIN
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Spectrogram
D = librosa.amplitude_to_db(np.abs(librosa.stft(y, hop_length=512)), ref=np.max)
librosa.display.specshow(D, sr=sr, hop_length=512, x_axis='time', y_axis='log', ax=axes[0])
axes[0].set_title('Spectrogram + PYIN Pitch Contour')
axes[0].plot(times_pyin, f0_pyin_voiced, color='cyan', linewidth=2, label='PYIN f0')
axes[0].set_ylim(50, 3000)
axes[0].legend(loc='upper right')

# Swara labels on time axis
semitones = []
for item in seq_pyin:
    if item and item.get('semitone') is not None:
        semitones.append(item['semitone'])
    else:
        semitones.append(np.nan)

axes[1].scatter(
    times_pyin, semitones,
    c=[SWARA_COLORS.get(int(s), '#888888') if not np.isnan(s) else '#888888' for s in semitones],
    s=10, alpha=0.8
)
axes[1].set_yticks(range(12))
axes[1].set_yticklabels([SWARAS_12TET[i][0] for i in range(12)])
axes[1].set_ylabel('Swara')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('PYIN — Swara Timeline')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Strategy 2 — CREPE (Deep Learning Pitch Estimator)

CREPE is a CNN trained on a large corpus of pitched audio. It gives per-frame confidence scores and is generally more accurate than YIN on real-world recordings.

In [ ]:
import crepe

# CREPE expects 16 kHz audio
y_16k = librosa.resample(y, orig_sr=sr, target_sr=16000)

# model='tiny'|'small'|'medium'|'large'|'full'
# step_size in ms (10ms gives very fine resolution)
time_crepe, frequency_crepe, confidence_crepe, _ = crepe.predict(
    y_16k, 16000, model='small', step_size=10, viterbi=True, verbose=0
)

# Mask low-confidence frames (threshold tunable)
CREPE_CONFIDENCE_THRESHOLD = 0.5
f0_crepe_masked = np.where(confidence_crepe >= CREPE_CONFIDENCE_THRESHOLD, frequency_crepe, np.nan)

seq_crepe = freqs_to_swara_sequence(f0_crepe_masked, time_crepe, sa_hz=SA_HZ)
runs_crepe = collapse_to_swara_run(seq_crepe)

voiced_count = np.sum(confidence_crepe >= CREPE_CONFIDENCE_THRESHOLD)
print(f'CREPE: {voiced_count} voiced frames / {len(confidence_crepe)} total (threshold={CREPE_CONFIDENCE_THRESHOLD})')
print('\nFirst 15 note events:')
for r in runs_crepe[:15]:
    dur = round(r['end'] - r['start'], 3) if r['end'] and r['start'] else '?'
    print(f"  {r['start']:.3f}s – {r['end']:.3f}s  [{dur}s]  → {r['swara']}")

In [ ]:
# Visualize CREPE
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(time_crepe, frequency_crepe, color='lightgray', linewidth=0.8, label='raw f0')
axes[0].plot(time_crepe, f0_crepe_masked, color='tomato', linewidth=1.5, label=f'f0 (conf≥{CREPE_CONFIDENCE_THRESHOLD})')
axes[0].fill_between(time_crepe, 0, confidence_crepe * max(frequency_crepe[~np.isnan(frequency_crepe)] or [1]),
                     alpha=0.15, color='orange', label='confidence (scaled)')
axes[0].set_ylabel('Frequency (Hz)')
axes[0].set_title('CREPE — Pitch Contour & Confidence')
axes[0].legend(loc='upper right')
axes[0].set_ylim(0, 2000)

semitones_crepe = []
for item in seq_crepe:
    if item and item.get('semitone') is not None:
        semitones_crepe.append(item['semitone'])
    else:
        semitones_crepe.append(np.nan)

axes[1].scatter(
    time_crepe, semitones_crepe,
    c=[SWARA_COLORS.get(int(s), '#888888') if not np.isnan(s) else '#888888' for s in semitones_crepe],
    s=8, alpha=0.7
)
axes[1].set_yticks(range(12))
axes[1].set_yticklabels([SWARAS_12TET[i][0] for i in range(12)])
axes[1].set_ylabel('Swara')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('CREPE — Swara Timeline')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Strategy 3 — Harmonic Product Spectrum (HPS)

A classic DSP approach. We multiply downsampled copies of the spectrum together — the true fundamental is reinforced while harmonics cancel each other out. Works well for monophonic wind instruments.

In [ ]:
def harmonic_product_spectrum_pitch(
    y, sr, hop_length=512, n_fft=4096, n_harmonics=5,
    fmin=80, fmax=2000
):
    """
    Estimate pitch frame-by-frame using Harmonic Product Spectrum.
    Returns (times, freqs) arrays.
    """
    # STFT
    stft = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

    # Frequency band limits
    f_low  = np.searchsorted(freqs, fmin)
    f_high = np.searchsorted(freqs, fmax)

    pitch_freqs = []
    for frame in stft.T:       # iterate over time frames
        hps = frame.copy()
        for h in range(2, n_harmonics + 1):
            downsampled = frame[::h][:len(hps)]
            hps[:len(downsampled)] *= downsampled

        # Find peak in [fmin, fmax] band
        band = hps[f_low:f_high]
        peak_idx = np.argmax(band) + f_low
        pitch_freqs.append(freqs[peak_idx])

    times = librosa.frames_to_time(
        np.arange(len(pitch_freqs)), sr=sr, hop_length=hop_length
    )
    return times, np.array(pitch_freqs)


# Compute energy-based voiced/unvoiced mask
def voiced_mask_energy(y, sr, hop_length=512, threshold_db=-40):
    rms = librosa.feature.rms(y=y, hop_length=hop_length)[0]
    rms_db = librosa.amplitude_to_db(rms, ref=np.max)
    return rms_db > threshold_db


times_hps, f0_hps = harmonic_product_spectrum_pitch(y, sr)
voiced_hps = voiced_mask_energy(y, sr)
f0_hps_voiced = np.where(voiced_hps[:len(f0_hps)], f0_hps, np.nan)

seq_hps  = freqs_to_swara_sequence(f0_hps_voiced, times_hps, sa_hz=SA_HZ)
runs_hps = collapse_to_swara_run(seq_hps)

print(f'HPS: {np.sum(~np.isnan(f0_hps_voiced))} voiced frames / {len(f0_hps_voiced)} total')
print('\nFirst 15 note events:')
for r in runs_hps[:15]:
    dur = round(r['end'] - r['start'], 3) if r['end'] and r['start'] else '?'
    print(f"  {r['start']:.3f}s – {r['end']:.3f}s  [{dur}s]  → {r['swara']}")

In [ ]:
# Visualize HPS
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(times_hps, f0_hps_voiced, color='mediumseagreen', linewidth=1.5, label='HPS f0')
axes[0].set_ylabel('Frequency (Hz)')
axes[0].set_title('HPS — Pitch Contour')
axes[0].legend()
axes[0].set_ylim(0, 2000)

semitones_hps = [
    item['semitone'] if item and item.get('semitone') is not None else np.nan
    for item in seq_hps
]
axes[1].scatter(
    times_hps, semitones_hps,
    c=[SWARA_COLORS.get(int(s), '#888888') if not np.isnan(s) else '#888888' for s in semitones_hps],
    s=10, alpha=0.8
)
axes[1].set_yticks(range(12))
axes[1].set_yticklabels([SWARAS_12TET[i][0] for i in range(12)])
axes[1].set_ylabel('Swara')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('HPS — Swara Timeline')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Strategy 4 — Basic-Pitch (Spotify)

Basic-Pitch is Spotify's polyphonic pitch estimation model. It exports MIDI events directly, which map cleanly to swaras. It handles ornaments (gamakas) and microtones better than pure pitch trackers for complex melodies.

In [ ]:
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

# Basic-Pitch returns three outputs: model output, MIDI, note events
model_output, midi_data, note_events = predict(AUDIO_FILE)

# note_events: list of (start_time, end_time, pitch_midi, amplitude, pitch_bends)
print(f'Basic-Pitch detected {len(note_events)} note events')
print('\nFirst 15 note events (MIDI → Swara):')

def midi_to_hz(midi_note):
    return 440.0 * (2 ** ((midi_note - 69) / 12))

runs_bp = []
for start, end, pitch, amp, bends in note_events[:15]:
    freq = midi_to_hz(pitch)
    swara = hz_to_swara(freq, sa_hz=SA_HZ)
    runs_bp.append({'swara': swara['short_name'] if swara else None, 'start': start, 'end': end})
    print(f"  {start:.3f}s – {end:.3f}s  MIDI={pitch}  {freq:.1f}Hz  → {swara['short_name'] if swara else '—'}")

In [ ]:
# Visualize Basic-Pitch piano-roll style
fig, ax = plt.subplots(figsize=(14, 5))

for start, end, pitch, amp, bends in note_events:
    freq  = midi_to_hz(pitch)
    swara = hz_to_swara(freq, sa_hz=SA_HZ)
    st    = swara['semitone'] if swara else 0
    color = SWARA_COLORS.get(st, '#888888')
    ax.barh(y=st, width=end - start, left=start, height=0.8, color=color, alpha=0.8)
    label = swara['short_name'] if swara else ''
    if (end - start) > 0.15:
        ax.text(start + (end - start) / 2, st, label,
                ha='center', va='center', fontsize=7, color='white', fontweight='bold')

ax.set_yticks(range(12))
ax.set_yticklabels([SWARAS_12TET[i][0] for i in range(12)])
ax.set_xlabel('Time (s)')
ax.set_title('Basic-Pitch — Swara Piano Roll')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Strategy 5 — Voting Ensemble

Combine PYIN, CREPE, and HPS predictions at each time step by majority vote. When all three agree, confidence is high. Disagreements flag ornamental (gamaka) regions.

In [ ]:
# Resample all swara sequences to a common time grid
from scipy.interpolate import interp1d

def resample_semitones(times_src, semitones_src, times_target):
    """Nearest-neighbour resample swara (semitone) sequence to a new time grid."""
    valid = ~np.isnan(semitones_src)
    if valid.sum() == 0:
        return np.full(len(times_target), np.nan)
    out = np.full(len(times_target), np.nan)
    for i, t in enumerate(times_target):
        idx = np.argmin(np.abs(times_src - t))
        if not np.isnan(semitones_src[idx]):
            out[i] = semitones_src[idx]
    return out


# Use PYIN time grid as reference
t_ref = times_pyin

st_pyin  = np.array([item['semitone'] if item and item.get('semitone') is not None else np.nan for item in seq_pyin])
st_crepe = resample_semitones(time_crepe, np.array(semitones_crepe, dtype=float), t_ref)
st_hps   = resample_semitones(times_hps,  np.array(semitones_hps,  dtype=float), t_ref)

# Majority vote
voted = []
agreement = []

for p, c, h in zip(st_pyin, st_crepe, st_hps):
    votes = [v for v in [p, c, h] if not np.isnan(v)]
    if not votes:
        voted.append(np.nan)
        agreement.append(0)
    else:
        most_common = Counter(votes).most_common(1)[0]
        voted.append(most_common[0])
        agreement.append(most_common[1])   # how many methods agreed

voted = np.array(voted)
agreement = np.array(agreement)

print(f'Ensemble: {np.sum(~np.isnan(voted))} voiced frames')
print(f'3-way agreement: {np.sum(agreement == 3)} frames ({100*np.mean(agreement[~np.isnan(voted)] == 3):.1f}%)')
print(f'2-way agreement: {np.sum(agreement == 2)} frames ({100*np.mean(agreement[~np.isnan(voted)] == 2):.1f}%)')

In [ ]:
# Visualize ensemble
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

labels = ['PYIN', 'CREPE (resampled)', 'HPS (resampled)', 'Ensemble Vote']
series = [st_pyin, st_crepe, st_hps, voted]
colors = ['steelblue', 'tomato', 'mediumseagreen', 'gold']

for ax, s, label, color in zip(axes, series, labels, colors):
    ax.scatter(t_ref, s, c=color, s=6, alpha=0.7)
    ax.set_yticks(range(12))
    ax.set_yticklabels([SWARAS_12TET[i][0] for i in range(12)], fontsize=8)
    ax.set_title(label)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(-0.5, 11.5)

# Shade disagreement regions on ensemble plot
axes[3].scatter(t_ref, voted,
    c=['gold' if a == 3 else 'orange' if a == 2 else '#aaaaaa' for a in agreement],
    s=8, alpha=0.9
)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Strategy Comparison — Frame-Level Swara Predictions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 8. Gamaka (Ornament) Detection

In Indian classical music, gamakas are pitch ornaments (meend, andolan, gamak). We detect them as frames where the pitch changes rapidly — the instantaneous pitch derivative is large.

In [ ]:
def detect_gamakas(f0, times, threshold_cents_per_sec=300.0):
    """
    Flag frames as gamaka (ornamental) when the pitch velocity
    exceeds `threshold_cents_per_sec`.

    Returns a boolean mask (True = gamaka frame).
    """
    valid = ~np.isnan(f0)
    cents = np.full_like(f0, np.nan)
    cents[valid] = 1200 * np.log2(f0[valid] / np.nanmin(f0[valid] + 1e-9))

    # Finite difference velocity (cents/sec)
    dt = np.diff(times, prepend=times[0])
    dcents = np.abs(np.gradient(cents, times))

    gamaka_mask = (dcents > threshold_cents_per_sec) & valid
    return gamaka_mask, dcents


gamaka_mask_pyin, pitch_velocity_pyin = detect_gamakas(f0_pyin_voiced, times_pyin)

n_gamaka = gamaka_mask_pyin.sum()
pct = 100 * n_gamaka / max(1, (~np.isnan(f0_pyin_voiced)).sum())
print(f'Gamaka frames (PYIN): {n_gamaka} ({pct:.1f}% of voiced frames)')

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(times_pyin, f0_pyin_voiced, color='steelblue', linewidth=1, label='PYIN f0', alpha=0.7)
ax.scatter(
    times_pyin[gamaka_mask_pyin], f0_pyin_voiced[gamaka_mask_pyin],
    color='red', s=15, zorder=5, label='Gamaka'
)
ax.set_ylabel('Frequency (Hz)')
ax.set_xlabel('Time (s)')
ax.set_title('Gamaka Detection — PYIN Pitch')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Swara Distribution & Raga Hint

After detecting swaras, we can compare the distribution of used swaras against known raga templates to suggest which raga the piece most closely resembles.

In [ ]:
# Swara usage distribution (from ensemble)
voiced_voted = voted[~np.isnan(voted)].astype(int)
swara_counts = Counter(voiced_voted)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(
    [SWARAS_12TET[i][0] for i in range(12)],
    [swara_counts.get(i, 0) for i in range(12)],
    color=[SWARA_COLORS[i] for i in range(12)],
    edgecolor='white'
)
ax.set_xlabel('Swara')
ax.set_ylabel('Frame count')
ax.set_title('Swara Usage Distribution (Ensemble)')
plt.tight_layout()
plt.show()

# Normalised swara presence vector (1 = present, 0 = absent)
swara_presence = np.array([1 if swara_counts.get(i, 0) > 0 else 0 for i in range(12)])
print('Used swaras:', [SWARAS_12TET[i][0] for i in range(12) if swara_presence[i]])

In [ ]:
# Simple raga matching by semitone set overlap
# Each raga is defined by its ascending (aroha) semitone set.
RAGA_TEMPLATES = {
    'Yaman':          [0, 2, 4, 6, 7, 9, 11],   # Kalyan thaat
    'Bhairav':        [0, 1, 4, 5, 7, 8, 11],   # Bhairav thaat
    'Bhoopali':       [0, 2, 4, 7, 9],           # Pentatonic (Kalyan)
    'Yaman Kalyan':   [0, 2, 4, 5, 6, 7, 9, 11],
    'Bhimpalasi':     [0, 2, 3, 5, 7, 9, 10],   # Kafi thaat
    'Darbari Kanada': [0, 2, 3, 5, 7, 8, 10],
    'Bageshri':       [0, 2, 3, 5, 7, 9, 10],
    'Malkauns':       [0, 3, 5, 8, 10],          # Pentatonic (Bhairavi)
    'Bhairavi':       [0, 1, 3, 5, 7, 8, 10],   # All komal
    'Kedar':          [0, 2, 4, 5, 6, 7, 9, 11],
    'Todi':           [0, 1, 3, 6, 7, 8, 11],
    'Marwa':          [0, 1, 4, 6, 9, 11],
    'Puriya Dhanashri': [0, 1, 4, 6, 7, 8, 11],
    'Hansadhwani':    [0, 2, 4, 7, 11],
    'Durga':          [0, 2, 5, 7, 9],
}

def raga_similarity(detected_swaras, raga_swaras):
    """Jaccard similarity between detected and raga swara sets."""
    det = set(detected_swaras)
    rag = set(raga_swaras)
    if not det and not rag:
        return 0.0
    return len(det & rag) / len(det | rag)


detected_set = set(i for i in range(12) if swara_presence[i])
scores = {
    raga: raga_similarity(detected_set, set(swaras))
    for raga, swaras in RAGA_TEMPLATES.items()
}
ranked = sorted(scores.items(), key=lambda x: -x[1])

print('Top 5 Raga matches (Jaccard similarity):')
for raga, score in ranked[:5]:
    print(f'  {raga:<25} {score:.3f}  |  notes: {[SWARAS_12TET[i][0] for i in RAGA_TEMPLATES[raga]]}')

# Bar chart
fig, ax = plt.subplots(figsize=(12, 4))
ragas_sorted  = [r for r, _ in ranked]
scores_sorted = [s for _, s in ranked]
ax.barh(ragas_sorted[::-1], scores_sorted[::-1], color='mediumpurple', edgecolor='white')
ax.set_xlabel('Jaccard Similarity')
ax.set_title('Raga Similarity Scores')
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='0.5 threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Export: Text Notation & MIDI

Produce a human-readable swara transcript and an exportable MIDI file.

In [ ]:
# Text notation from PYIN ensemble runs
ensemble_seq = []
for t, s in zip(t_ref, voted):
    if np.isnan(s):
        ensemble_seq.append({'time': float(t), 'short_name': None, 'semitone': None})
    else:
        info = SWARAS_12TET[int(s)]
        ensemble_seq.append({'time': float(t), 'short_name': info[0], 'semitone': int(s)})

runs_ensemble = collapse_to_swara_run(ensemble_seq)

print('Swara Transcript (Ensemble):')
print('=' * 50)
line = ''
for r in runs_ensemble:
    if r['swara']:
        dur = r['end'] - r['start']
        # Represent duration by repetition (roughly)
        count = max(1, int(round(dur / 0.25)))
        line += ' ' + (r['swara'] + '-' * (count - 1))
    else:
        line += ' |'
print(line.strip())

# Save to file
with open('swara_transcript.txt', 'w') as f:
    f.write('Swara Transcript (Ensemble — PYIN+CREPE+HPS)\n')
    f.write(f'Tonic Sa = {SA_HZ} Hz\n')
    f.write('=' * 50 + '\n\n')
    f.write(line.strip() + '\n\n')
    f.write('\nDetailed note events:\n')
    for r in runs_ensemble:
        if r['swara']:
            dur = r['end'] - r['start']
            f.write(f"{r['start']:.3f}s – {r['end']:.3f}s  [{dur:.3f}s]  {r['swara']}\n")

print('\nTranscript saved → swara_transcript.txt')

In [ ]:
# Export MIDI (uses Basic-Pitch MIDI output, or build from PYIN)
try:
    # If Basic-Pitch ran successfully, just save its MIDI
    midi_data.write('output_indian_notes.mid')
    print('Basic-Pitch MIDI saved → output_indian_notes.mid')
except NameError:
    # Fall back to building MIDI from PYIN + pretty_midi
    !pip install pretty_midi --quiet
    import pretty_midi

    pm = pretty_midi.PrettyMIDI()
    flute = pretty_midi.Instrument(program=73)  # program 73 = Flute

    for r in runs_ensemble:
        if r['swara'] and r['start'] is not None and r['end'] is not None:
            st = [k for k, v in SWARAS_12TET.items() if v[0] == r['swara']]
            if st:
                sa_midi = pretty_midi.hz_to_note_number(SA_HZ)
                midi_note_num = sa_midi + st[0]
                note = pretty_midi.Note(
                    velocity=80,
                    pitch=int(midi_note_num),
                    start=r['start'],
                    end=r['end'],
                )
                flute.notes.append(note)

    pm.instruments.append(flute)
    pm.write('output_indian_notes.mid')
    print('PYIN-based MIDI saved → output_indian_notes.mid')

## 11. Strategy Accuracy Comparison (with Ground Truth)

If you have a ground-truth swara annotation (e.g., from Tabla-Net or a manually labelled `.csv`), this cell computes frame-level accuracy for each strategy.

In [ ]:
# Replace with your ground-truth CSV path:
# Expected columns: time, swara_semitone  (semitone offset from Sa, 0–11)
GT_CSV = None   # e.g. 'ground_truth.csv'

if GT_CSV:
    import pandas as pd
    gt = pd.read_csv(GT_CSV)
    gt_times = gt['time'].values
    gt_semi  = gt['swara_semitone'].values.astype(float)

    def frame_accuracy(pred_times, pred_semi, gt_times, gt_semi):
        correct = 0
        total   = 0
        for t, g in zip(gt_times, gt_semi):
            if np.isnan(g):
                continue
            idx = np.argmin(np.abs(pred_times - t))
            p = pred_semi[idx]
            total += 1
            if not np.isnan(p) and int(p) == int(g):
                correct += 1
        return correct / max(1, total)

    acc_pyin  = frame_accuracy(t_ref, st_pyin,  gt_times, gt_semi)
    acc_crepe = frame_accuracy(t_ref, st_crepe, gt_times, gt_semi)
    acc_hps   = frame_accuracy(t_ref, st_hps,   gt_times, gt_semi)
    acc_ens   = frame_accuracy(t_ref, voted,    gt_times, gt_semi)

    print('Strategy Accuracy (frame-level):')
    print(f'  PYIN     : {acc_pyin:.3f}')
    print(f'  CREPE    : {acc_crepe:.3f}')
    print(f'  HPS      : {acc_hps:.3f}')
    print(f'  Ensemble : {acc_ens:.3f}')

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(['PYIN', 'CREPE', 'HPS', 'Ensemble'],
           [acc_pyin, acc_crepe, acc_hps, acc_ens],
           color=['steelblue', 'tomato', 'mediumseagreen', 'gold'])
    ax.set_ylabel('Frame Accuracy')
    ax.set_ylim(0, 1)
    ax.set_title('Strategy Comparison — Frame Accuracy vs Ground Truth')
    plt.tight_layout()
    plt.show()
else:
    print('Set GT_CSV to a ground-truth annotation file to run accuracy comparison.')

## 12. Summary & Tuning Guide

| Strategy | Strengths | Weaknesses | Best for |
|----------|-----------|------------|----------|
| **PYIN** | Fast, voiced/unvoiced decisions, no GPU | Can struggle with breathy flute attacks | Clean studio recordings |
| **CREPE** | High accuracy on real audio, viterbi smoothing | Needs GPU for speed, 10ms latency | Complex, ornamented passages |
| **HPS** | Pure DSP, no ML dependencies, interpretable | Sensitive to noise, octave errors | Live/noisy audio analysis |
| **Basic-Pitch** | Polyphonic, exports MIDI, handles ornaments | Heavier model, slower | MIDI export, accompaniment separation |
| **Ensemble** | Most robust, flags disagreement regions | Slowest (runs all three) | Final production output |

### Tuning the Tonic (Sa)
- Use `librosa.yin` or `librosa.pyin` on the drone/tanpura track to auto-detect Sa.
- Or use `librosa.effects.pitch_shift` to normalise the recording to C4 (261.63 Hz) before processing.

### Handling Gamakas
- Set `gap_tolerance_s` in `collapse_to_swara_run()` larger (e.g. 0.1s) to bridge gamaka micro-fluctuations.
- For meend (glide), use the raw pitch contour rather than quantised swaras.

### Improving Raga Recognition
- Weight swaras by duration, not just presence.
- Add aroha/avaroha (ascending/descending) directionality.
- Use a trained classifier (SVM / LSTM) on the swara duration histogram.